# TP 2 — Construire le pipeline et ouvrir l'enquête

**Durée : 2 h.** À partir du semestre complet, produire des tables fiables puis trouver un comportement inhabituel. Vous ne connaissez à l'avance ni le réseau concerné, ni la date, ni la cause.

Lire `TP2_PIPELINE.md`. Les cellules d'installation et de collecte sont fournies pour pouvoir reprendre sur un nouveau noyau. Toutes les transformations et l'enquête se font en SQL. Aucun graphique n'est demandé ici : les visualisations seront faites dans Power BI au TP 3.

In [ ]:
%pip install -q "duckdb==1.4.3"

In [ ]:
from pathlib import Path
import duckdb
from IPython.core.magic import register_cell_magic
from IPython.display import display

# Chemins relatifs : les fichiers sont sur la machine qui exécute le noyau.
Path("data").mkdir(exist_ok=True)
Path("exports").mkdir(exist_ok=True)
if "db" in globals():
    db.close()
db = duckdb.connect("data/atelier.duckdb")

# Petit utilitaire fourni : il permet d'écrire des cellules SQL.
@register_cell_magic
def sql(line, cell):
    if cell.strip() and any(not x.strip().startswith("--") and x.strip() for x in cell.splitlines()):
        result = db.execute(cell)
        if result.description:
            display(result.fetch_df())

print("Dossier d'exécution :", Path.cwd())

In [ ]:
from urllib.request import urlretrieve

url = "https://www.eia.gov/electricity/gridmonitor/sixMonthFiles/EIA930_BALANCE_2024_Jan_Jun.csv"
fichier = Path("data/eia930_2024_s1.csv")

if not fichier.exists():
    temporaire = fichier.with_suffix(".part")
    urlretrieve(url, temporaire)
    temporaire.replace(fichier)

print("Fichier prêt :", fichier, "—", round(fichier.stat().st_size / 1_000_000, 1), "Mo")

In [ ]:
%%sql
CREATE OR REPLACE VIEW brut AS
SELECT *
FROM read_csv('data/eia930_2024_s1.csv', header = true, all_varchar = true);

SELECT * FROM brut LIMIT 5;

## 1. Profiler avant de transformer

Écrire un bilan : lignes par réseau, absences et valeurs négatives pour demande, production totale et production solaire. Vérifier les doublons de la clé réseau + heure UTC.

Colonnes utiles : `"UTC Time at End of Hour"`, `"Demand (MW)"`, `"Net Generation (MW)"`, `"Net Generation (MW) from Solar"`.

Les champs sont du texte : `TRY_CAST(... AS DOUBLE)` permet de tester une conversion. Distinguer champ absent et texte non vide impossible à convertir.

In [ ]:
%%sql
-- À vous de coder : bilan des valeurs manquantes, négatives et non convertibles.

In [ ]:
%%sql
-- À vous de coder : chercher les doublons par réseau et heure UTC.

## 2. Créer la table typee

Créer `typee` avec les colonnes ci-dessous. Conserver toutes les lignes et toutes les valeurs : le classement qualité vient après.

| Source | Colonne cible | Type |
|---|---|---|
| Balancing Authority | reseau | VARCHAR |
| Data Date | date_jour | DATE |
| UTC Time at End of Hour | fin_utc | TIMESTAMP |
| Local Time at End of Hour | fin_locale | TIMESTAMP |
| Demand (MW) | demande_mw | DOUBLE |
| Net Generation (MW) | production_mw | DOUBLE |
| Total Interchange (MW) | export_net_mw | DOUBLE |
| Net Generation (MW) from Solar | solaire_mw | DOUBLE |
| Net Generation (MW) from Natural Gas | gaz_mw | DOUBLE |
| Net Generation (MW) from Coal | charbon_mw | DOUBLE |
| Net Generation (MW) from Nuclear | nucleaire_mw | DOUBLE |
| Net Generation (MW) from Wind | eolien_mw | DOUBLE |
| Net Generation (MW) from Hydropower and Pumped Storage | hydraulique_mw | DOUBLE |
| Net Generation (MW) from Other Fuel Sources | autres_mw | DOUBLE |
| Net Generation (MW) from All Petroleum Products | petrole_mw | DOUBLE |
| Net Generation (MW) from Unknown Fuel Sources | inconnu_mw | DOUBLE |

Les dates sont au format américain : `%m/%d/%Y`. Les heures utilisent `%m/%d/%Y %I:%M:%S %p`. Utiliser `TRY_STRPTIME` puis un cast en DATE si nécessaire.

Les colonnes suffixées « Adjusted » ou « Imputed » ne doivent pas être mélangées aux valeurs déclarées dans cette première version.

In [ ]:
%%sql
-- À vous de coder : CREATE OR REPLACE TABLE typee AS SELECT ... FROM brut;

## 3. Qualifier, conserver et vérifier

Créer `qualifiee` en ajoutant `statut_solaire` :
- `date_invalide` si la date ou un horodatage ne se convertit pas ;
- `non_renseigne` si la mesure solaire est absente ;
- `net_negatif_a_interpreter` si la mesure solaire est négative ;
- `exploitable` sinon.

Ce statut concerne l'analyse solaire, pas la validité de toute la ligne. Une production nette négative n'est pas nécessairement une erreur ; ne pas la remplacer arbitrairement par zéro. Une absence de demande peut concerner un réseau qui ne la déclare pas.

Créer `reseau_horaire` avec les dates valides en conservant les statuts et les autres mesures, et `rejets_dates` pour les dates invalides. Compter les lignes par statut et vérifier la conservation du nombre total de lignes.

In [ ]:
%%sql
-- À vous de coder : créer qualifiee, reseau_horaire et rejets_dates ; afficher le bilan.

Contrôler maintenant les intervalles entre deux mesures pour chaque réseau, triées en UTC. Ne pas imposer 24 heures à chaque journée locale : le semestre contient un changement d'heure. Comparer les décalages entre heures locales et UTC sur plusieurs réseaux et plusieurs dates.

In [ ]:
%%sql
-- À vous de coder : calculer le pas UTC entre deux mesures et inspecter les pas différents de 1 heure.

## 4. Enquêter

Créer `variations` : toutes les colonnes de `reseau_horaire`, le pas UTC, la variation solaire et la variation gaz par rapport à la mesure précédente du même réseau.

Calculer les fenêtres sur toute la série avant de filtrer. Ne comparer que des heures réellement consécutives et des mesures solaires exploitables aux deux extrémités. Il faut donc aussi conserver le statut précédent.

Classer les baisses observées en journée. Justifier votre plage horaire et examiner plusieurs candidats : une valeur extrême seule ne donne pas sa cause.

In [ ]:
%%sql
-- À vous de coder : créer variations avec LAG partitionné par réseau et ordonné par fin_utc.

In [ ]:
%%sql
-- À vous de coder : classer les baisses diurnes sur des heures consécutives et des valeurs exploitables.

Choisir un candidat. Examiner les heures avant et après, puis les autres filières, la demande et les échanges sur le même créneau. Comparer à quelques journées voisines. Écrire deux hypothèses avant toute recherche de l'explication sur Internet.

In [ ]:
%%sql
-- À vous de coder : examiner le candidat choisi, les autres filières et plusieurs jours de comparaison.

## 5. Préparer Power BI

Créer `dim_temps` avec une ligne par combinaison réseau/heure : `cle_heure`, `reseau`, `date_jour`, `fin_utc`, `fin_locale`, `statut_solaire`.

Clé proposée : `reseau || '|' || CAST(fin_utc AS VARCHAR)`.

Créer `fact_production` avec une ligne par réseau/heure/filière : `cle_heure`, `filiere`, `puissance_mw`. Utiliser `UNPIVOT INCLUDE NULLS` sur les neuf colonnes de filières. Conserver les absences. Ne pas ajouter demande et production totale parmi les filières : cela compterait deux fois des quantités.

Vérifier : clé unique dans la dimension, neuf lignes de faits par heure, aucune clé de faits sans correspondance.

In [ ]:
%%sql
-- À vous de coder : créer dim_temps et fact_production au grain demandé.

In [ ]:
%%sql
-- À vous de coder : contrôler les clés, le nombre de lignes et la jointure.

## 6. Exporter
Exécuter après avoir créé les tables. Les exports seront sur la machine du noyau. Suivre le support pour les récupérer sur le poste Power BI.

In [ ]:
%%sql
COPY dim_temps TO 'exports/dim_temps.csv' (HEADER, DELIMITER ';');
COPY fact_production TO 'exports/fact_production.csv' (HEADER, DELIMITER ';');
COPY reseau_horaire TO 'exports/reseau_horaire.csv' (HEADER, DELIMITER ';');

## Note d'enquête à compléter

- Réseau et créneau retenus : …
- Trois observations chiffrées : …
- Règles de qualité appliquées et effet sur les lignes : …
- Deux hypothèses : …
- Ce que le dashboard devra permettre de vérifier : …
- Une limite des données : …

Enregistrer, committer et être prêt à modifier une requête sur demande. La conclusion sera confrontée aux visuels au TP 3.